# Assignment 3.2: Factory production optimization

This notebook adapts the PuLP workflow in the provided `501_M3_lab.ipynb` to the factory problem. The goal is to choose production quantities and whether to run Machine 2 on Saturday to maximize weekly profit.

## 1. Decision variables and objective

- $x$: units of product A per week, continuous and nonnegative.
- $y$: units of product B per week, continuous and nonnegative.
- $z$: binary variable, equal to 1 when Machine 2 runs on Saturday and 0 otherwise.

Maximize net weekly profit:

$$\max\; 40x + 60y - 500z$$

The first two terms are product profits; the final term charges $500 only when overtime is used.

## 2. Constraints

$$\begin{aligned}
2x + 4y &\le 100 && \text{Machine 1 hours}\\
3x + 2y &\le 80 + 20z && \text{Machine 2 hours, including optional overtime}\\
y &\ge 10 && \text{minimum production of B}\\
x,y &\ge 0, \quad z \in \{0,1\} && \text{variable domains}
\end{aligned}$$

For instance, each A uses 3 Machine 2 hours, and each B uses 2. Setting $z=1$ increases that machine's available time from 80 to 100 hours.

## 3. Build and solve with PuLP

The lab starts by installing and importing PuLP. HiGHS is installed here as the mixed-integer solver; the installation cell can be skipped when both packages are already available.

In [1]:
%pip install "pulp<4" highspy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
    --------------------------------------- 0.3/16.4 MB ? eta -:--:--
   ---------------------------- ----------- 11.5/16.4 MB 30.1 MB/s eta 0:00:01
   ---------------------------------------- 16.4/16.4 MB 35.6 MB/s eta 0:00:00
  Attempting uninstall: pulp
    Found existing installation: PuLP 4.2.0
    Uninstalling PuLP-4.2.0:
      Successfully uninstalled PuLP-4.2.0
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import pulp

factory = pulp.LpProblem("Factory_Weekly_Profit", pulp.LpMaximize)

# A and B are continuous as specified; overtime is a yes/no decision.
x = pulp.LpVariable("A", lowBound=0, cat="Continuous")
y = pulp.LpVariable("B", lowBound=0, cat="Continuous")
z = pulp.LpVariable("Saturday_overtime", cat="Binary")

factory += 40*x + 60*y - 500*z, "Net_weekly_profit"
factory += 2*x + 4*y <= 100, "Machine_1_hours"
factory += 3*x + 2*y <= 80 + 20*z, "Machine_2_hours"
factory += y >= 10, "Minimum_B"
print(factory)

Factory_Weekly_Profit:
MAXIMIZE
40*A + 60*B + -500*Saturday_overtime + 0
SUBJECT TO
Machine_1_hours: 2 A + 4 B <= 100

Machine_2_hours: 3 A + 2 B - 20 Saturday_overtime <= 80

Minimum_B: B >= 10

VARIABLES
A Continuous
B Continuous
0 <= Saturday_overtime <= 1 Integer



In [3]:
status = factory.solve(pulp.HiGHS(msg=False))
print("Status:", pulp.LpStatus[status])
if pulp.LpStatus[status] != "Optimal":
    raise RuntimeError("No optimal solution was found; inspect the model and solver status.")

print(f"Product A: {x.value():.2f} units")
print(f"Product B: {y.value():.2f} units")
print(f"Saturday overtime: {int(round(z.value()))} (1 = yes, 0 = no)")
print(f"Maximum weekly net profit: ${pulp.value(factory.objective):,.2f}")

Status: Optimal
Product A: 15.00 units
Product B: 17.50 units
Saturday overtime: 0 (1 = yes, 0 = no)
Maximum weekly net profit: $1,650.00


## 4. Check resource use and compare the overtime choices

The following checks calculate actual hours used and solve the model again with each overtime choice fixed. The comparison includes the $500 cost in the overtime scenario.

In [4]:
print(f"Machine 1: {2*x.value() + 4*y.value():.2f} / 100 hours")
print(f"Machine 2: {3*x.value() + 2*y.value():.2f} / {80 + 20*z.value():.2f} hours")
print(f"Product B quota: {y.value():.2f} / 10 minimum units")

for overtime in (0, 1):
    scenario = factory.copy()
    scenario += z == overtime, f"Fix_overtime_{overtime}"
    scenario_status = scenario.solve(pulp.HiGHS(msg=False))
    if pulp.LpStatus[scenario_status] != "Optimal":
        raise RuntimeError(f"Scenario {overtime}: {pulp.LpStatus[scenario_status]}")
    print(f"Overtime = {overtime}: A = {x.value():.2f}, B = {y.value():.2f}, "
          f"net profit = ${pulp.value(scenario.objective):,.2f}")

Machine 1: 100.00 / 100 hours
Machine 2: 80.00 / 80.00 hours
Product B quota: 17.50 / 10 minimum units
Overtime = 0: A = 15.00, B = 17.50, net profit = $1,650.00
Overtime = 1: A = 25.00, B = 12.50, net profit = $1,250.00


## 5. Interpretation

The optimal weekly plan is **15 units of A and 17.5 units of B**, giving **$1,650** in profit without Saturday overtime. It uses all 100 hours on Machine 1 and all 80 regular hours on Machine 2, and exceeds the minimum B quota by 7.5 units.

With overtime forced on, the best plan produces 25 units of A and 12.5 units of B. Its gross product profit is $1,750, but after the $500 overtime charge its net profit is $1,250. The additional machine time improves gross profit by $100, less than its $500 cost, so Saturday overtime should not be run. The question's phrase “Machine B on overtime” is interpreted as Machine 2, as described in the problem.

The assignment explicitly allows continuous production, so 17.5 units is valid in this model. If products must be whole units in practice, the variables should instead be integers and the model solved again. Other real-world limits, such as demand and material availability, are not included in the provided data.

## AI assistance disclosure

OpenAI ChatGPT (Codex) assisted in adapting the provided lab notebook, formulating and solving the model, and drafting the explanations. Review the code, verify the calculations, and revise this disclosure to accurately describe your own use and any changes before submitting. Cite the tool in the format required by your course.